In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

In [ ]:
# Explorar os dados
df = pd.read_csv('transacoes_fraude.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['Tipo de Transacao'].value_counts()

In [ ]:
df['Classe'].value_counts()

In [ ]:
# Preparar os dados
df.isnull().sum()

In [ ]:
df = df.dropna()
df['Hora'] = pd.to_datetime(df['Horario da Transacao']).dt.hour
df = pd.get_dummies(df, columns=['Tipo de Transacao'])
df = df.drop(columns=['Cliente', 'Horario da Transacao'])
df.head()

In [ ]:
# Dividir em treino e teste
X = df.drop(columns=['Classe'])
y = df['Classe']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Treinar a árvore de decisão
modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

In [ ]:
# Sensibilidade e especificidade
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print('Sensibilidade:', tp / (tp + fn))
print('Especificidade:', tn / (tn + fp))

In [ ]:
pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False)

## Análise dos resultados

O modelo teve sensibilidade de cerca de 13% e especificidade de cerca de 88%. Ou seja, ele deixa passar quase 9 em cada 10 fraudes, e ainda marca como fraude perto de 12% das transações legítimas. Não chegou ao equilíbrio que o banco procura.

A base é desbalanceada (só 11% das transações são fraude), e as variáveis disponíveis parecem não separar bem as duas classes. A árvore usou mais os valores da transação e do saldo, mas o resultado ficou próximo de um chute, o que indica que essas variáveis sozinhas não carregam um padrão claro de fraude.

Possíveis melhorias: balancear as classes (class_weight ou reamostragem), limitar a profundidade da árvore para reduzir o overfitting, criar novas variáveis a partir do histórico de cada cliente (frequência e valor médio das transações) e testar outros modelos, como Random Forest. Mesmo assim, se os dados não tiverem relação real com a fraude, nenhum modelo vai resolver.